In [7]:
from langchain.document_loaders import CSVLoader

In [8]:
filepath = "Assignments/assignment2dataset.csv"

In [9]:
loader = CSVLoader(filepath, encoding = 'utf-8')
documents = loader.load()

In [13]:
from dotenv import load_dotenv

load_dotenv()

True

In [37]:
model = "text-embedding-3-large"
llm_model = "gpt-4.1-mini"

In [33]:
from langchain_openai import AzureOpenAIEmbeddings
embeddings = AzureOpenAIEmbeddings(model=model,api_version="2024-12-01-preview")
from langchain_openai import AzureChatOpenAI
llm = AzureChatOpenAI(model=llm_model,api_version="2024-12-01-preview")

In [36]:
from langchain_chroma import Chroma
vectorstore = Chroma.from_documents(documents=documents, embedding=embeddings)

In [ ]:
## Implementation ##

In [57]:
from typing import TypedDict, Literal, List
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langgraph.graph import StateGraph,START,END

retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 2})
message = """
Answer this question using the provided context only. If the context does not have the content to answer the question, say context is irrelevant.
Do not make up the answer, do not provide answer from outside the doc.
{question}

Context:
{context}
"""

ragprompt = PromptTemplate.from_template(message)

class ragState(TypedDict):
    question: str
    context: List[Document]
    answer: str

def retriever_node(state:ragState):
    ret_docs = retriever.invoke(state["question"])
    return {"context":ret_docs}

# generation node
def generate(state:ragState):
    doc_content = "\n\n".join(doc.page_content for doc in state["context"])
    message = ragprompt.invoke({"question":state["question"],"context":doc_content})
    response = llm.invoke(message)
    return {"answer":response}

builder = StateGraph(ragState).add_sequence([retriever_node,generate])
builder.add_edge(START,"retriever_node")
builder.add_edge("generate",END)
raggraph = builder.compile()   

response = raggraph.invoke({"question":"what are the courses if I have interest in Vizualization ?"})

In [58]:
output = response

In [60]:
print(response['answer'].content)


The course related to visualization is:

- Data Visualization with Tableau (course_id: C014)
